In [2]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [3]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [ ]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_fraud_experimentation.json")

In [5]:
from pg_schema.loader import SchemaLoader

source_schema_path = "../../dtgraph/pg_schema/schemas/schema_fraud_source.json"
target_schema_path = "../../dtgraph/pg_schema/schemas/schema_fraud_target.json"

source_schema = SchemaLoader(
    source_schema_path,
    env=env,
    section="source"
)

target_schema = SchemaLoader(
    target_schema_path,
    env=env,
    section="target"
)

##### Rules

In [6]:
Rule1 = Rule(
    """
MATCH (c:Client)
WHERE NOT c:Mule
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 3000
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
""",
    env=env,
    type_strict=True,
)

Rule2 = Rule(
    """
MATCH (c:Client:Mule)
OPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)
OPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)
OPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)
WITH 
    c,
    collect(DISTINCT e.email) AS emails,
    collect(DISTINCT p.phoneNumber) AS phones,
    collect(DISTINCT s.ssn) AS ssns
LIMIT 3000
GENERATE
(p = (c.id):Person,Scammer {
    id = c.id,
    name = c.name,
    name_camel_case = apoc.text.upperCamelCase(c.name),
    email = head(emails),
    phone = head(phones),
    ssn = head(ssns)
})
""",
    env=env,
    type_strict=True,
)

Rule3 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:CashIn)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_CASHIN]->(
    agg = (c.id, "cashin"):CashInSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule4 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:CashOut)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_CASHOUT]->(
    agg = (c.id, "cashout"):CashOutSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule5 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:Transfer)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_TRANSFER]->(
    agg = (c.id, "transfer"):TransferSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule6 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:Payment)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_PAYMENT]->(
    agg = (c.id, "payment"):PaymentSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule7 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t:Debit)
WITH c, collect(t.amount) AS amounts
GENERATE
(p = (c.id):Person {
    id = c.id,
    name = c.name
})
-[():AGGREGATED_DEBIT]->(
    agg = (c.id, "debit"):DebitSummary {
        all_amounts = amounts,
        count = size(amounts)
    }
)
""",
env=env,
type_strict=True,
)

Rule8 = Rule(
"""
MATCH (c:Client)-[:PERFORMED]->(t1:Transaction)-[:NEXT]->(t2:Transaction)
OPTIONAL MATCH (t2)-[:TO]->(target)
WITH c, t1, t2, target
GENERATE
(f = (c.id, t1.id, t2.id):TransactionFlow {
    from_amount = t1.amount,
    to_amount = t2.amount,
    increasing = t2.amount > t1.amount
})
-[():STARTED_BY]-> (p = (c.id):),
(f)-[():FROM_TX]-> (tx1 = (t1.id):NewTransaction {
    id = t1.id,
    amount = t1.amount
}),
(f)-[():TO_TX]-> (tx2 = (t2.id):NewTransaction {
    id = t2.id,
    amount = t2.amount
}),
(f)-[():TARGET]-> (dest = (target.id):Destination)
""",
env=env,
type_strict=True,
)


--- Checking Rule ---
{'lhs': 'MATCH (c:Client)\nWHERE NOT c:Mule\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWITH \n    c,\n    collect(DISTINCT e.email) AS emails,\n    collect(DISTINCT p.phoneNumber) AS phones,\n    collect(DISTINCT s.ssn) AS ssns\nLIMIT 3000', 'constructors': [{'alias': 'p', 'ids': ['c.id'], 'labels': ['Person'], 'properties': [{'key': 'id', 'value': 'c.id'}, {'key': 'name', 'value': 'c.name'}, {'key': 'name_camel_case', 'value': 'apoc.text.upperCamelCase(c.name)'}, {'key': 'email', 'value': 'head(emails)'}, {'key': 'phone', 'value': 'head(phones)'}, {'key': 'ssn', 'value': 'head(ssns)\n'}]}]}

 Type checking passed


--- Checking Rule ---
{'lhs': 'MATCH (c:Client:Mule)\nOPTIONAL MATCH (c)-[:HAS_EMAIL]->(e:Email)\nOPTIONAL MATCH (c)-[:HAS_PHONE]->(p:Phone)\nOPTIONAL MATCH (c)-[:HAS_SSN]->(s:SSN)\nWITH \n    c,\n    collect(DISTINCT e.email) AS emails,\n    collect(DISTINCT p.phone

##### Schema Conformance

In [ ]:
from dtgraph.pg_schema.check_schema import check_schema

check_schema(
    [
        Rule1,
        # # Rule2,
        # # Rule3,
        # # Rule4,
        # # Rule5,
        # # Rule6,
        # # Rule7,
        # Rule8
    ],
    target_schema,
)

##### Applying Rules

In [7]:
my_transform = Transformation(
    [
        # Rule1,
        # Rule2,
        # Rule3,
        # Rule4,
        # Rule5,
        # Rule6,
        # Rule7,
        Rule8
    ]
)
my_transform.apply_on(graph)

Index: Added 1 index, completed after 63 ms.
Rule: Added 1293859 labels, created 646930 nodes, set 4179657 properties, created 1284628 relationships, completed after 28458 ms.


28458

##### Abort Transformation

In [8]:
my_transform.abort()

Index: Removed 1 index, completed after 15 ms.


Abort: Deleted 646930 nodes, deleted 1284628 relationships, completed after 4334 ms.
